In [2]:
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
 
DAVIS = Path("data_raw/DAVIS")
JPEG = DAVIS / "JPEGImages" / "480p"
ANNO = DAVIS / "Annotations" / "480p"
 
assert JPEG.exists(), f"not found: {JPEG} (run from app/, after the download script)"
clips = sorted([d.name for d in JPEG.iterdir() if d.is_dir()])
print(f"{len(clips)} clips")
print(clips)

AssertionError: not found: data_raw\DAVIS\JPEGImages\480p (run from app/, after the download script)

In [ ]:
rows = []
for c in clips:
  n_jpg = len(list((JPEG / c).glob("*.jpg")))
  n_png = len(list((ANNO / c).glob("*.png")))
  rows.append((c, n_jpg, n_png, "OK" if n_jpg == n_png else "MISMATCH"))
 
print(f"{'clip':<22}{'frames':>8}{'masks':>8}  status")
for c, j, p, s in rows:
  print(f"{c:<22}{j:>8}{p:>8}  {s}")
mismatches = [r for r in rows if r[3] != "OK"]
print(f"\n{len(mismatches)} mismatched clips" if mismatches else "\nall clips: frames == masks")
counts = [j for _, j, _, _ in rows]
print(f"frames per clip: min={min(counts)} max={max(counts)} mean={np.mean(counts):.1f}")

In [ ]:
def mask_indices(clip, frame_idx=0):
  p = sorted((ANNO / clip).glob("*.png"))[frame_idx]
  arr = np.array(Image.open(p))          # paletted -> 2D array of indices (mode 'P')
  return arr, np.unique(arr)
 
example = clips[0]
arr, vals = mask_indices(example)
print(f"clip '{example}' frame 0 mask:")
print(f"  shape={arr.shape}  dtype={arr.dtype}  unique values={vals}")
print(f"  -> {'BINARY (single object)' if set(vals.tolist()) <= {0,1} else f'MULTI-INSTANCE ({len(vals)-1} objects + bg)'}")

multi = []
for c in clips:
  _, v = mask_indices(c)
  n_obj = len([x for x in v if x != 0])
  if n_obj > 1:
    multi.append((c, n_obj))
print(f"\n{len(multi)} clips have >1 object instance:")
for c, n in multi:
  print(f"  {c}: {n} objects")
print("\nNOTE: for single-object trajectory, pick ONE instance per clip (e.g. index 1),")
print("or treat 'any nonzero' as the object if you want all instances merged.")

In [ ]:
areas = []
for c in clips:
  arr, _ = mask_indices(c, 0)
  areas.append(float((arr > 0).mean()))
areas = np.array(areas)
print(f"mask area fraction: min={areas.min():.3f} max={areas.max():.3f} "
      f"mean={areas.mean():.3f} median={np.median(areas):.3f}")
 
plt.figure(figsize=(7, 4))
plt.hist(areas, bins=20, edgecolor="black")
plt.axvline(0.01, color="r", ls="--", label="min_area_frac=0.01")
plt.axvline(0.60, color="orange", ls="--", label="max_area_frac=0.60")
plt.xlabel("mask area fraction (frame 0)"); plt.ylabel("# clips")
plt.title("DAVIS object size distribution"); plt.legend(); plt.tight_layout()
plt.show()
# clips outside the dashed band would be filtered by your PairConfig area gates.
print("clips below 0.01:", [clips[i] for i in range(len(clips)) if areas[i] < 0.01])
print("clips above 0.60:", [clips[i] for i in range(len(clips)) if areas[i] > 0.60])

In [ ]:
def centroid(arr):
  ys, xs = np.nonzero(arr > 0)
  if len(xs) == 0: return None
  return (xs.mean(), ys.mean())
 
disps = []
for c in clips:
  pngs = sorted((ANNO / c).glob("*.png"))
  a0 = np.array(Image.open(pngs[0])); aN = np.array(Image.open(pngs[-1]))
  c0, cN = centroid(a0), centroid(aN)
  if c0 and cN:
    d = ((cN[0]-c0[0])**2 + (cN[1]-c0[1])**2) ** 0.5
    disps.append((c, d))
 
disps.sort(key=lambda x: x[1], reverse=True)
print("centroid displacement frame0 -> frameN (480p px), most-moving first:")
for c, d in disps[:12]:
  print(f"  {c:<22}{d:8.1f}")
print("  ...")
for c, d in disps[-5:]:
  print(f"  {c:<22}{d:8.1f}")
dvals = np.array([d for _, d in disps])
plt.figure(figsize=(7, 4))
plt.hist(dvals, bins=20, edgecolor="black")
plt.xlabel("centroid displacement (px, full clip)"); plt.ylabel("# clips")
plt.title("DAVIS object motion magnitude"); plt.tight_layout(); plt.show()

In [ ]:
def overlay(clip, frame_idx):
  pngs = sorted((ANNO / clip).glob("*.png"))
  jpgs = sorted((JPEG / clip).glob("*.jpg"))
  img = np.array(Image.open(jpgs[frame_idx]).convert("RGB"))
  m = (np.array(Image.open(pngs[frame_idx])) > 0)
  out = img.copy()
  out[m] = (0.5 * out[m] + 0.5 * np.array([255, 0, 0])).astype(np.uint8)  # red tint on object
  return out
 
sample = clips[:6]
fig, axes = plt.subplots(len(sample), 2, figsize=(8, 3 * len(sample)))
for r, c in enumerate(sample):
  n = len(list((JPEG / c).glob("*.jpg")))
  axes[r, 0].imshow(overlay(c, 0));     axes[r, 0].set_title(f"{c}  frame 0");     axes[r, 0].axis("off")
  axes[r, 1].imshow(overlay(c, n - 1)); axes[r, 1].set_title(f"{c}  frame {n-1}"); axes[r, 1].axis("off")
plt.tight_layout(); plt.show()